# BEC stability

This notebook follows a synthetic sequence of BEC-like absorption images from raw records to fitted observables. The prepared data are loaded directly into an ordinary pandas DataFrame, independently of any laboratory loading backend.

Features demonstrated:

- compact inspection and subtree selection of flattened fields;
- vectorized image normalization, image moments, and two-dimensional BEC fitting through `.m`;
- image tables, radial profiles, and stored fit overlays through `.p`;
- quick visualization of scalar results with time traces, histograms, and pair plots.

## Import

In [ ]:
import numpy as np
import pandas as pd

import labdata

%load_ext labdata.notebook

In [ ]:
with np.load("data/demo1.npz", allow_pickle=False) as d:
    DATA = pd.DataFrame({
        "FILE": d["FILE"],
        "TIME": d["TIME"],
        "IMAGE": list(d["IMAGE"]),
        "METADATA//CAMERA": d["METADATA//CAMERA"],
        "METADATA//ROI": [tuple(roi) for roi in d["METADATA//ROI"]],
    })[d["__columns__"].tolist()]

DATA.head()

## Raw data

`%print` presents nested fields and array summaries compactly. The image table lays out the records in time order, exposing changes in the cloud and background.

In [ ]:
%print DATA

In [ ]:
DATA.p.table(x=["TIME"], width=100, ncols=15)
DATA.p.imshow(vmin=0, vmax=1, cmap="turbo")

## Background subtraction

`imagenorm(border=3)` estimates a constant background from the border pixels of each image and subtracts it. The normalized image table shows the result across the complete sequence.

In [ ]:
d = DATA.m.imagenorm(border=3)
d.p.table(x=["TIME"], width=100, ncols=15)
d.p.imshow(vmin=0, vmax=1, cmap="turbo")

## Atom-number estimate

The `pixelsum` method uses image moments rather than an assumed cloud shape. For every record it calculates the integrated signal, centre, and width, and stores those derived values under `var`. In this notebook, the `pixelsum` (integrated signal) is referred to as the atom number below.

In [ ]:
d = d.m.pixelsum()

In [ ]:
%print d

Flattened paths are ordinary DataFrame columns. Selecting the exact field name therefore returns familiar pandas and NumPy values that can be inspected or passed to other tools directly.

In [ ]:
d["var//pixelsum"].values

## Atom-number stability and distribution

The `plot` and `hist` methods provide quick views of the data as a time trace and a distribution.

In [ ]:
d.p.plot(x="TIME", y="var//pixelsum")

In [ ]:
d.p.hist(x="var//pixelsum", bins=12)

## BEC fit

`bec2dfit` applies the two-dimensional BEC model independently to every normalized image. It preserves the source fields and adds fitted values, uncertainties, diagnostics, and the complete numerical result under explicit field paths.

The fitted DataFrame stores scalar fit values under `var` and the complete numerical fit under `xy_fit`. Selecting `xy_fit//popt` returns a DataFrame restricted to the optimized fit parameters for each image.

In [ ]:
d = DATA.m.imagenorm(border=3)
d = d.m.bec2dfit()

In [ ]:
%print d

In [ ]:
%print d["xy_fit//popt"]

## Fit inspection

`implot` can draw horizontal (`X`), vertical (`Y`), summed (`XS` and `YS`), or radial (`R`) image profiles and overlay a stored fit result without refitting the data. This example uses the radial profile.

In [ ]:
d.p.table(x=["TIME"], width=100, ncols=12)
d.p.implot("R", fit="xy_fit")

## Cloud position and correlations

The fitted horizontal centre is an ordinary scalar column and can be plotted directly against `TIME`.

In [ ]:
d.p.plot(x="TIME", y="var//x_centre[pixel]")

`pairplot` provides a compact view of the distributions and correlations between selected scalar fields.

Here, the atom number (`pixelsum`) and fitted height are correlated, as expected, while the horizontal position appears uncorrelated with atom number.

In [ ]:
d.p.pairplot(vars=["var//pixelsum", "var//height", "var//x_centre[pixel]"])